In [1]:
import io
import os
import boto3
import duckdb
import pandas as pd
from datasets import load_dataset


In [2]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())

before = int(con.sql("SELECT COUNT(*) FROM raw.coco_annotations").fetchone()[0])
snap_before = int(con.sql("FROM ducklake_snapshots('lake')").df().iloc[-1]["snapshot_id"])
print(f"Rows before: {before}")
print(f"Last snapshot before: {snap_before}")


Rows before: 36781
Last snapshot before: 19


In [3]:
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
BUCKET = "lakehouse"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
print("S3 client ready.")


S3 client ready.


In [4]:
print("Loading 100 samples from train split...")
train_ds = load_dataset("detection-datasets/coco", split="train", streaming=True)

rows = []
for i, sample in enumerate(train_ds):
    if i >= 100:
        break

    image_id = sample["image_id"]
    image = sample["image"]

    buf = io.BytesIO()
    image.save(buf, format="JPEG")
    buf.seek(0)
    key = f"assets/coco/images/{image_id}.jpg"
    s3.upload_fileobj(buf, BUCKET, key)
    image_uri = f"s3://{BUCKET}/{key}"

    objects = sample["objects"]
    for j in range(len(objects["bbox_id"])):
        rows.append({
            "image_uri": image_uri,
            "image_id": image_id,
            "width": sample["width"],
            "height": sample["height"],
            "bbox_id": objects["bbox_id"][j],
            "category": objects["category"][j],
            "bbox": str(objects["bbox"][j]),
            "area": objects["area"][j],
        })

    if (i + 1) % 10 == 0:
        print(f"{i + 1}/100 images processed")

print(f"New annotation rows to insert: {len(rows)}")


Loading 100 samples from train split...


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

10/100 images processed
20/100 images processed
30/100 images processed
40/100 images processed
50/100 images processed
60/100 images processed
70/100 images processed
80/100 images processed
90/100 images processed
100/100 images processed
New annotation rows to insert: 785


In [5]:
new_df = pd.DataFrame(rows)
con.execute("INSERT INTO raw.coco_annotations SELECT * FROM new_df")
print("Inserted.")

Inserted.


In [6]:
after = int(con.sql("SELECT COUNT(*) FROM raw.coco_annotations").fetchone()[0])
snap_after = int(con.sql("FROM ducklake_snapshots('lake')").df().iloc[-1]["snapshot_id"])

print(f"Rows before: {before}  →  after: {after}  (+{after - before})")
print(f"Snapshot before: {snap_before}  →  after: {snap_after}")
print(con.sql("FROM ducklake_snapshots('lake')").df().tail(3))

Rows before: 36781  →  after: 37566  (+785)
Snapshot before: 19  →  after: 20
    snapshot_id                    snapshot_time  schema_version  \
18           18 2026-06-26 03:08:27.281367+00:00              18   
19           19 2026-06-26 03:08:27.305056+00:00              19   
20           20 2026-06-26 03:54:20.960096+00:00              19   

                                              changes author commit_message  \
18                         {'tables_dropped': ['14']}   None           None   
19  {'tables_created': ['gold.coco_training'], 'ta...   None           None   
20                    {'tables_inserted_into': ['4']}   None           None   

   commit_extra_info  
18              None  
19              None  
20              None  


In [7]:
con.close()